# Final Pipeline Persistence with Pickle

This notebook persists the final fitted SECOM model artifact using Python `pickle` and validates that loading it produces identical predictions, Fail probabilities, and decision threshold.

## Contents

1. [Goal](#1-goal)
2. [Data and Final Pipeline](#2-data-and-final-pipeline)
3. [Fit and Save Artifact](#3-fit-and-save-artifact)
4. [Round-Trip Validation](#4-round-trip-validation)
5. [Persistence Notes](#5-persistence-notes)

## 1. Goal

The saved pickle contains a complete artifact dictionary, not only the classifier:

```python
final_model_artifact = {
    "pipeline": fitted_pipeline,
    "fail_threshold": selected_threshold,
    "max_missing_fraction": 0.50,
    "positive_label": 1,
    "negative_label": -1,
    "model_name": final_model_name,
    "selection_metric": threshold_selection_metric,
}
```

Persisting the threshold is necessary because the final decision logic uses a custom Fail threshold of `0.35`, not the default `0.50` cutoff.

In [1]:
# Imports
from pathlib import Path
import pickle
import importlib.metadata as metadata
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import sklearn
import imblearn
import mlflow

from sklearn.model_selection import train_test_split

from src.data_loading import load_secom_data
from src.evaluation import predict_with_threshold, summarize_model
from src.final_model import (
    FINAL_FAIL_THRESHOLD,
    FINAL_MODEL_NAME,
    FINAL_RANDOM_STATE,
    build_final_model_artifact,
    build_final_pipeline,
    load_final_model_artifact,
    save_final_model_artifact,
)

## 2. Data and Final Pipeline

The same stratified train/test split is used for consistency with the main notebook. The final pipeline is fitted on the full training set only.

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
X, y = load_secom_data(DATA_DIR)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=FINAL_RANDOM_STATE,
)

best_pipeline = build_final_pipeline()

print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Final model: {FINAL_MODEL_NAME}")
print(f"Final Fail threshold: {FINAL_FAIL_THRESHOLD:.2f}")


Training shape: (1253, 590)
Test shape: (314, 590)
Final model: RF + SelectKBest tuned
Final Fail threshold: 0.35


## 3. Fit and Save Artifact

The fitted pipeline and decision threshold are saved together under `models/secom_best_pipeline.pkl` with the highest pickle protocol.

In [3]:
models_dir = PROJECT_ROOT / "models"
model_path = models_dir / "secom_best_pipeline.pkl"

best_pipeline.fit(X_train, y_train)
final_model_artifact = build_final_model_artifact(best_pipeline)

pred_before, proba_before = predict_with_threshold(
    final_model_artifact["pipeline"],
    X_test,
    final_model_artifact["fail_threshold"],
    positive_label=final_model_artifact["positive_label"],
    negative_label=final_model_artifact["negative_label"],
)
threshold_before = final_model_artifact["fail_threshold"]

save_final_model_artifact(final_model_artifact, model_path)

model_size_mb = model_path.stat().st_size / (1024 * 1024)
print(f"Saved model artifact to: {model_path.relative_to(PROJECT_ROOT)}")
print(f"Pickle protocol used: {pickle.HIGHEST_PROTOCOL}")
print(f"Model artifact file size: {model_size_mb:.2f} MB")


Saved model artifact to: models\secom_best_pipeline.pkl
Pickle protocol used: 5
Model artifact file size: 0.71 MB


## 4. Round-Trip Validation

The validation confirms that predictions, Fail probabilities, and the persisted threshold are identical after loading the pickle file.

In [4]:
loaded_artifact = load_final_model_artifact(model_path)

pred_after, proba_after = predict_with_threshold(
    loaded_artifact["pipeline"],
    X_test,
    loaded_artifact["fail_threshold"],
    positive_label=loaded_artifact["positive_label"],
    negative_label=loaded_artifact["negative_label"],
)
threshold_after = loaded_artifact["fail_threshold"]

assert np.array_equal(pred_before, pred_after)
assert np.allclose(proba_before, proba_after)
assert threshold_before == threshold_after

round_trip_metrics = summarize_model(
    f"{loaded_artifact['model_name']} threshold {threshold_after:.2f}",
    y_test,
    pred_after,
    proba_after,
)

print("Round-trip validation passed.")
print(f"Persisted threshold: {threshold_after:.2f}")
round_trip_metrics

Round-trip validation passed.
Persisted threshold: 0.35


{'model': 'RF + SelectKBest tuned threshold 0.35',
 'accuracy': 0.9012738853503185,
 'balanced_accuracy': 0.6376564277588168,
 'fail_precision': 0.2916666666666667,
 'fail_recall': 0.3333333333333333,
 'fail_f1': 0.3111111111111111,
 'roc_auc': 0.7705184462863643,
 'average_precision': 0.2382166891465322}

## 5. Persistence Notes

The pickle file contains the fitted preprocessing pipeline and classifier together with the custom decision threshold. Inference should therefore load the artifact dictionary and apply:

```python
y_pred = np.where(y_fail_proba >= artifact["fail_threshold"], 1, -1)
```

Security and compatibility notes:

- pickle files should only be loaded from trusted sources;
- loading a pickle can execute code during deserialization;
- the loading environment should use compatible Python and library versions;
- the model was created for this project environment, not as a portable production artifact.

In [5]:
environment_packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "imbalanced-learn",
    "matplotlib",
    "seaborn",
    "jupyter",
    "notebook",
    "ipykernel",
    "mlflow",
]

environment_versions = pd.DataFrame(
    [{"package": "python", "version": sys.version.split()[0]}]
    + [
        {"package": package, "version": metadata.version(package)}
        for package in environment_packages
    ]
)

environment_path = PROJECT_ROOT / "environment_versions.txt"
environment_versions.assign(line=lambda df: df["package"] + "==" + df["version"])['line'].to_csv(
    environment_path,
    index=False,
    header=False,
)

environment_versions

,package,version
0,python,3.13.7
1,numpy,2.4.6
2,pandas,2.3.3
3,scipy,1.17.1
4,scikit-learn,1.9.0
5,imbalanced-learn,0.14.2
6,matplotlib,3.11.0
7,seaborn,0.13.2
8,jupyter,1.1.1
9,notebook,7.5.7
